# 008 — Intent Weight Tuning

Tune `intent_weights` for `intent_experiment` **without re-running CLIP / SpLiCE**.

## How it works
1. Run `main.py` once with `--cache-path <some.pt>` (see `scripts/exp.sh`) to dump per-query similarity components: `pred_sim`, `inst_sim`, `ref_sim`, `hd_sim` (and `mod_flag`, query intent labels, target/index metadata).
2. This notebook loads the cache and re-computes mAP/recall purely from the cached similarity tensors, so each weight evaluation is a single weighted sum + topk (ms per call).
3. Coarse grid search → focused Optuna refinement.

Required: `pip install optuna pandas` (the rest already in the project env).

In [1]:
import os
from itertools import product
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd
import torch

# Path to cache produced by:
#   python main.py ... --cache-path precomputed/intent_components_cache.pt
CACHE_PATH = '/workspace/joel/HDCIR/HDCIReVL/logs/circo_GPT_cache.pt'
OBJECTIVE  = 'mAP@5'           # primary metric to optimize
RETRIEVALS = (5, 10, 25, 50)

INTENT_NAMES = [
    'negation', 'addition', 'direct_addressing', 'compare_change',
    'spatial_relations_background', 'viewpoint', 'comparative_statement', 'cardinality',
]

assert os.path.exists(CACHE_PATH), f'Cache not found: {CACHE_PATH}\nRun main.py with --cache-path first.'

## 1. Load cache

The cache stores per-query `[Q, N]` similarity vectors (float16) plus everything needed to compute mAP/recall.

In [2]:
cache = torch.load(CACHE_PATH, map_location='cpu')

# Keep matmul in fp32 for stability — the cache is fp16 only for disk size.
pred_sims = cache['pred_sims'].float()   # [Q, N]
inst_sims = cache['inst_sims'].float()
ref_sims  = cache['ref_sims'].float()
hd_sims   = cache['hd_sims'].float()      # zeros where mod_flag is False
mod_flags = cache['mod_flags']             # [Q] bool
query_labels = cache['query_labels']       # list of list[str]
target_names = cache['target_names']
targets       = cache['targets']
index_names   = np.array(cache['index_names'])

Q, N = pred_sims.shape
print(f'queries={Q}, index_size={N}, fraction with HD modification={mod_flags.float().mean().item():.2%}')
print(f'captured active_intents = {cache.get("active_intents_at_capture")}')
print(f'captured intent_weights = {cache.get("intent_weights_at_capture")}')

queries=220, index_size=123403, fraction with HD modification=69.09%
captured active_intents = ['negation', 'addition', 'direct_addressing', 'compare_change', 'spatial_relations_background', 'viewpoint', 'comparative_statement', 'cardinality']
captured intent_weights = {'negation': (1.0, 0.0, 0.0, 1.0), 'addition': (1.0, 0.4, 0.0, 0.0), 'direct_addressing': (1.0, 0.44999999999999996, 0.0, 0.5), 'compare_change': (1.0, 0.4, 0.0, 1.0), 'spatial_relations_background': (1.0, 0.1, 0.0, 0.0), 'viewpoint': (1.0, 0.5, 0.0, 1.0), 'comparative_statement': (1.0, 0.25, 0.0, 0.0), 'cardinality': (1.0, 0.1, 0.0, 0.0), 'default': (1.0, 0.0, 0.0, 0.0)}


## 2. Fast evaluator

`evaluate(intent_weights, active_intents)` reproduces the metric loop from `intent_experiment` but vectorized: a single 4-term weighted sum across `[Q, N]`, then per-query topk.

In [3]:
DEFAULT_INTENT_VEC = (1.0, 0.0, 0.0, 0.0)

# Pre-compute per-query target sets and the float index needed for mAP precision.
_targets_clean = [np.array([t for t in row if t != '']) for row in targets]
_target_names_np = np.array(target_names)
_position_denoms = torch.arange(1, 51, dtype=torch.float32)


def build_per_query_weights(intent_weights: Dict[str, Tuple[float, float, float, float]],
                            active_intents: List[str]) -> torch.Tensor:
    """Per-query [Q, 4] weight matrix, mirroring the averaging-over-labels logic in intent_experiment."""
    active_set = set(active_intents)
    default = intent_weights.get('default', DEFAULT_INTENT_VEC)
    W = torch.zeros(Q, 4)
    for i, labels in enumerate(query_labels):
        acc = [0.0, 0.0, 0.0, 0.0]
        count = 0
        for lbl in labels:
            if lbl in intent_weights and lbl in active_set:
                for j in range(4):
                    acc[j] += intent_weights[lbl][j]
                count += 1
        if count == 0:
            acc = list(default)
        else:
            acc = [a / count for a in acc]
        W[i] = torch.tensor(acc)
    # ratio-preserving scaling on w2 (same as production code).
    W[:, 1] = W[:, 1] * (W[:, 0] + W[:, 2] + W[:, 3])
    return W


def evaluate(intent_weights: Dict[str, Tuple[float, float, float, float]],
             active_intents: List[str],
             retrievals=RETRIEVALS) -> Dict[str, float]:
    W = build_per_query_weights(intent_weights, active_intents)
    sim = (W[:, 0:1] * pred_sims
           + W[:, 1:2] * inst_sims
           + W[:, 2:3] * ref_sims
           + W[:, 3:4] * hd_sims)

    topk_vals, topk_idx = sim.topk(k=50, dim=-1)
    topk_names = index_names[topk_idx.numpy()]   # [Q, 50]

    maps    = {k: [] for k in retrievals}
    recalls = {k: [] for k in retrievals}
    for q in range(Q):
        sub_targets = _targets_clean[q]
        names_q = topk_names[q]
        map_labels = np.isin(names_q, sub_targets).astype(np.float32)
        precisions = np.cumsum(map_labels) * map_labels / np.arange(1, len(map_labels) + 1)
        for k in retrievals:
            maps[k].append(precisions[:k].sum() / min(len(sub_targets), k))
        gt_hits = (names_q == _target_names_np[q]).astype(np.float32)
        for k in retrievals:
            recalls[k].append(gt_hits[:k].sum())

    out = {f'mAP@{k}': float(np.mean(maps[k])) * 100 for k in retrievals}
    out.update({f'recall@{k}': float(np.mean(recalls[k])) * 100 for k in retrievals})
    return out

## 3. Sanity check — production weights

Reproduce the metrics that the captured run produced. If this differs from the log, the cache is stale.

In [4]:
PRODUCTION_WEIGHTS = {
    'negation':                     (1.0, 0.0,  0.0, 1.0),
    'addition':                     (1.0, 0.4,  0.0, 0.0),
    'direct_addressing':            (1.0, 0.3,  0.0, 0.5),
    'compare_change':               (1.0, 0.2,  0.0, 1.0),
    'spatial_relations_background': (1.0, 0.1,  0.0, 0.0),
    'viewpoint':                    (1.0, 0.25, 0.0, 1.0),
    'comparative_statement':        (1.0, 0.25, 0.0, 0.0),
    'cardinality':                  (1.0, 0.1,  0.0, 0.0),
    'default':                      (1.0, 0.0,  0.0, 0.0),
}
ACTIVE_INTENTS = INTENT_NAMES

baseline = evaluate(PRODUCTION_WEIGHTS, ACTIVE_INTENTS)
for k, v in baseline.items():
    print(f'  {k:>12} = {v:6.3f}')

         mAP@5 = 17.005
        mAP@10 = 18.202
        mAP@25 = 20.245
        mAP@50 = 21.055
      recall@5 = 28.182
     recall@10 = 39.091
     recall@25 = 58.182
     recall@50 = 66.818


## 4. Coarse grid search — one intent at a time

Vary `(w2, w4)` for a single intent while holding the rest at production. Cheap (a few hundred evaluations).

In [5]:
def sweep_one_intent(target_intent: str,
                     w2_grid=(0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0),
                     w4_grid=(0.0, 0.1, 0.2, 0.3, 0.4, 0.5),
                     base_weights=PRODUCTION_WEIGHTS,
                     active_intents=ACTIVE_INTENTS):
    rows = []
    w1, w2_base, w3_base, w4_base = base_weights[target_intent]
    for w2, w4 in product(w2_grid, w4_grid):
        weights = {k: tuple(v) for k, v in base_weights.items()}
        weights[target_intent] = (w1, w2, w3_base, w4)
        m = evaluate(weights, active_intents)
        rows.append({'intent': target_intent, 'w2': w2, 'w4': w4, **m})
    return pd.DataFrame(rows)


# Example: tune `negation` (which currently uses w4=1.0)
df_neg = sweep_one_intent('negation')
df_neg.sort_values(OBJECTIVE, ascending=False).head(10)

,intent,w2,w4,mAP@5,mAP@10,mAP@25,mAP@50,recall@5,recall@10,recall@25,recall@50
9,negation,0.1,0.3,17.294444,18.457024,20.456708,21.280256,29.545453,39.545456,58.181816,66.818184
4,negation,0.0,0.4,17.289899,18.440971,20.432451,21.256256,29.545453,39.545456,58.181816,66.818184
10,negation,0.1,0.4,17.267172,18.420949,20.412438,21.235397,29.545453,39.545456,58.181816,66.818184
6,negation,0.1,0.0,17.262121,18.446017,20.454669,21.285104,29.545453,39.545456,59.090906,67.272729
12,negation,0.2,0.0,17.262121,18.445228,20.445736,21.281908,29.545453,39.545456,58.636361,67.272729
22,negation,0.3,0.4,17.248990,18.435217,20.427238,21.245979,29.545453,39.545456,58.181816,67.272729
18,negation,0.3,0.0,17.246212,18.430392,20.430823,21.258830,29.545453,39.545456,58.636361,67.272729
5,negation,0.0,0.5,17.234091,18.368911,20.369945,21.193206,29.545453,39.545456,58.181816,66.818184
21,negation,0.3,0.3,17.230808,18.416368,20.409117,21.233227,29.545453,39.545456,58.181816,67.272729
20,negation,0.3,0.2,17.230808,18.419975,20.420541,21.252053,29.545453,39.545456,58.181816,67.272729


In [6]:
# Sweep every intent in sequence; report each intent's best (w2, w4) under that-intent-only changes.
best_per_intent = {}
for intent in INTENT_NAMES:
    df = sweep_one_intent(intent)
    best = df.sort_values(OBJECTIVE, ascending=False).iloc[0]
    best_per_intent[intent] = best
    print(f"{intent:>32}  w2={best['w2']:.2f}  w4={best['w4']:.2f}  {OBJECTIVE}={best[OBJECTIVE]:.3f}  (baseline {baseline[OBJECTIVE]:.3f})")

                        negation  w2=0.10  w4=0.30  mAP@5=17.294  (baseline 17.005)
                        addition  w2=0.60  w4=0.00  mAP@5=17.058  (baseline 17.005)
               direct_addressing  w2=0.80  w4=0.00  mAP@5=18.970  (baseline 17.005)
                  compare_change  w2=0.80  w4=0.10  mAP@5=18.130  (baseline 17.005)
    spatial_relations_background  w2=0.80  w4=0.00  mAP@5=17.430  (baseline 17.005)
                       viewpoint  w2=0.50  w4=0.00  mAP@5=17.773  (baseline 17.005)
           comparative_statement  w2=1.00  w4=0.20  mAP@5=17.194  (baseline 17.005)
                     cardinality  w2=0.50  w4=0.00  mAP@5=17.132  (baseline 17.005)


## 5. Compose grid-search winners

Naively applying every per-intent winner ignores interactions, but it's a strong starting point for Optuna.

In [7]:
GRID_BEST_WEIGHTS = {k: tuple(v) for k, v in PRODUCTION_WEIGHTS.items()}
for intent, row in best_per_intent.items():
    w1, _, w3, _ = PRODUCTION_WEIGHTS[intent]
    GRID_BEST_WEIGHTS[intent] = (w1, float(row['w2']), w3, float(row['w4']))

composed = evaluate(GRID_BEST_WEIGHTS, ACTIVE_INTENTS)
print('composed grid winners:')
for k, v in composed.items():
    arrow = '↑' if v > baseline[k] else ('↓' if v < baseline[k] else '=')
    print(f'  {k:>12} = {v:6.3f}   ({arrow} vs baseline {baseline[k]:.3f})')

for intent in INTENT_NAMES:
    print(f'  {intent:>32}: {GRID_BEST_WEIGHTS[intent]}')

composed grid winners:
         mAP@5 = 18.879   (↑ vs baseline 17.005)
        mAP@10 = 19.690   (↑ vs baseline 18.202)
        mAP@25 = 21.615   (↑ vs baseline 20.245)
        mAP@50 = 22.421   (↑ vs baseline 21.055)
      recall@5 = 29.091   (↑ vs baseline 28.182)
     recall@10 = 38.182   (↓ vs baseline 39.091)
     recall@25 = 52.727   (↓ vs baseline 58.182)
     recall@50 = 64.545   (↓ vs baseline 66.818)
                          negation: (1.0, 0.1, 0.0, 0.3)
                          addition: (1.0, 0.6, 0.0, 0.0)
                 direct_addressing: (1.0, 0.8, 0.0, 0.0)
                    compare_change: (1.0, 0.8, 0.0, 0.1)
      spatial_relations_background: (1.0, 0.8, 0.0, 0.0)
                         viewpoint: (1.0, 0.5, 0.0, 0.0)
             comparative_statement: (1.0, 1.0, 0.0, 0.2)
                       cardinality: (1.0, 0.5, 0.0, 0.0)


## 6. Optuna — joint search over all intents

Seeds the TPE study with the production point and the composed grid winners so it can't do worse.

In [ ]:
import optuna
from optuna.samplers import TPESampler

TUNED_INTENTS = INTENT_NAMES  # set to a smaller subset to fix some intents and tune the rest
W2_RANGE = (0.0, 1.0)
W3_RANGE = (0.0, 1.0)
W4_RANGE = (0.0, 2.0)


def _params_to_weights(params: Dict[str, float]) -> Dict[str, Tuple[float, float, float, float]]:
    w = {k: tuple(v) for k, v in PRODUCTION_WEIGHTS.items()}
    for intent in TUNED_INTENTS:
        w1 = 1.0  # never tune w1 — it scales the primary CLIP retrieval term.
        w2 = params[f'{intent}__w2']
        w3 = params[f'{intent}__w3']
        w4 = params[f'{intent}__w4']
        w[intent] = (w1, w2, w3, w4)
    return w


def objective(trial: optuna.Trial) -> float:
    params = {}
    for intent in TUNED_INTENTS:
        params[f'{intent}__w2'] = trial.suggest_float(f'{intent}__w2', *W2_RANGE)
        params[f'{intent}__w3'] = trial.suggest_float(f'{intent}__w3', *W3_RANGE)
        params[f'{intent}__w4'] = trial.suggest_float(f'{intent}__w4', *W4_RANGE)
    weights = _params_to_weights(params)
    m = evaluate(weights, ACTIVE_INTENTS)
    # Report secondary metrics for trial logs.
    for k, v in m.items():
        if k != OBJECTIVE:
            trial.set_user_attr(k, v)
    return m[OBJECTIVE]


def _weights_to_params(weights):
    return {f'{intent}__w{j}': weights[intent][j - 1]
            for intent in TUNED_INTENTS
            for j in (2, 3, 4)}


study = optuna.create_study(direction='maximize', sampler=TPESampler(seed=42, multivariate=True))
study.enqueue_trial(_weights_to_params(PRODUCTION_WEIGHTS))
study.enqueue_trial(_weights_to_params(GRID_BEST_WEIGHTS))
study.optimize(objective, n_trials=300, show_progress_bar=True)

print(f'best {OBJECTIVE} = {study.best_value:.3f}  (baseline {baseline[OBJECTIVE]:.3f})')

In [ ]:
best_weights = _params_to_weights(study.best_params)
best_metrics = evaluate(best_weights, ACTIVE_INTENTS)

print('full metrics for Optuna best:')
for k, v in best_metrics.items():
    arrow = '↑' if v > baseline[k] else ('↓' if v < baseline[k] else '=')
    print(f'  {k:>12} = {v:6.3f}   ({arrow} vs baseline {baseline[k]:.3f})')

print('\nbest intent_weights (paste into compute_results.py):')
print('intent_weights = {')
for intent in INTENT_NAMES:
    w1, w2, w3, w4 = best_weights[intent]
    print(f"    {intent!r:>34}: ({w1:.3f}, {w2:.3f}, {w3:.3f}, {w4:.3f}),")
print(f"    {'default'!r:>34}: {best_weights['default']},")
print('}')

## 7. Inspect the search

Top trials + parameter importance (mostly useful when one or two weights dominate).

In [ ]:
trials_df = study.trials_dataframe(attrs=('number', 'value', 'params', 'user_attrs'))
trials_df.sort_values('value', ascending=False).head(15)

In [ ]:
try:
    importances = optuna.importance.get_param_importances(study)
    for k, v in sorted(importances.items(), key=lambda kv: -kv[1])[:15]:
        print(f'  {k:>40}  {v:.3f}')
except Exception as e:
    print('importance failed (need >=2 finished trials with different params):', e)